# Teste isolado — ABEGÁS (Notícias do Setor)

Fonte candidata: **ABEGÁS - Associação Brasileira das Empresas Distribuidoras
de Gás Canalizado**, setor Energia/Gás. Notebook **descartável** (Fase 1) —
sem dispatcher, sem `atualizar_status_fonte`, sem gravar nada. Só valida:

1. Scraping da listagem (categoria, título, data, link)
2. Extração do texto completo de uma notícia individual

## Confirmado antes de assumir: WordPress, mas NÃO é o loop padrão

`robots.txt` confirma WordPress (`Disallow: /wp-admin/`, `Sitemap:
wp-sitemap.xml`), sem bloqueio pra `/noticias-do-setor`. Mas o tema usa um
page builder (Elementor + layout "packery/isotope" de um tema GT3) — os
seletores não são os genéricos de WordPress (`article`, `.entry-content`),
são específicos do tema: `div.blog_post_preview.noticias` na listagem,
`.single_meta .blog_content` no detalhe. Paginação é a padrão do WP
(`/noticias-do-setor/page/N`).

A API REST do WP (`/wp-json/wp/v2/categories?slug=noticias`) revelou
**14.274 posts** na categoria "Notícias" — histórico enorme. Por isso o
`max_paginas` fica conservador (5 páginas = 50 itens mais recentes por
execução), acompanhando o fluxo novo em vez de tentar arquivo histórico
completo — mesmo critério já usado em ANA/ANP.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
import unicodedata
import urllib.parse
from datetime import datetime
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://www.abegas.org.br/noticias-do-setor"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
ITENS_POR_PAGINA_ESPERADO = 10

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação)

`div.blog_post_preview.noticias` -> título/link em `h2.blogpost_title a`,
data em `span.post_date` (DD/MM/AAAA), categoria em `span.post_category a`.
Paginação: `?` nenhuma — é path (`/page/N`), diferente do `?b_start:int=N`
da ANA/ANP (aquilo é Plone; isso aqui é WordPress).

In [0]:
def listar_abegas(max_paginas: int = 5) -> list[dict]:
    itens = []

    for pagina in range(1, max_paginas + 1):
        url_pagina = SITE_URL if pagina == 1 else f"{SITE_URL.rstrip('/')}/page/{pagina}"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página {pagina}; parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        itens_pagina = soup.select("div.blog_post_preview.noticias")
        if not itens_pagina:
            print(f"  -> nenhum item encontrado na página {pagina}; fim da listagem.")
            break

        for item in itens_pagina:
            tag_a = item.select_one("h2.blogpost_title a")
            if not tag_a:
                continue
            categoria = item.select_one("span.post_category a")
            data = item.select_one("span.post_date")

            data_publicacao = None
            if data:
                m = re.match(r"(\d{2})/(\d{2})/(\d{4})", data.get_text(strip=True))
                if m:
                    dia, mes, ano = m.groups()
                    data_publicacao = f"{ano}-{mes}-{dia}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": tag_a["href"].strip(),
                "categoria": categoria.get_text(strip=True) if categoria else None,
                "published_at": data_publicacao,
            })

        print(f"  página {pagina}: {len(itens_pagina)} itens.")
        if len(itens_pagina) < ITENS_POR_PAGINA_ESPERADO:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_abegas()

print(f"\n{len(itens)} notícias listadas.\n")
print(f"{'DATA':<12} {'CATEGORIA':<18} TÍTULO")
print("-" * 100)
for item in itens:
    categoria = (item["categoria"] or "?")[:16]
    print(f"{item['published_at'] or '?':<12} {categoria:<18} {item['titulo'][:60]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]
sem_categoria = [i for i in itens if not i["categoria"]]

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)} | sem categoria: {len(sem_categoria)}")
print(f"\nExemplo de link: {itens[0]['url']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

`.single_meta .blog_content` — não é nenhum dos seletores já genéricos do
dispatcher (`#content-core`, `#parent-fieldname-text`, `#content`, `main`,
`.field--name-body`); precisa de um seletor próprio pra esse tema.

In [0]:
def extrair_texto_abegas(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    base = soup.select_one(".single_meta .blog_content") or soup.select_one(".blog_content")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_titulo_h1(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    return h1.get_text(" ", strip=True) if h1 else None


def extrair_noticia_abegas(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_abegas(html)
    titulo = extrair_titulo_h1(html) or item["titulo"]

    return {
        "titulo": titulo,
        "url": item["url"],
        "categoria": item["categoria"],
        "published_at": item["published_at"],
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_abegas(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos.")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars: {len(curtas)}")

In [0]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"CATEGORIA   : {detalhe['categoria']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem paginada extrai categoria/título/data/link
de todos os itens, texto completo sai limpo com um seletor específico do
tema (`.single_meta .blog_content`).

**Avaliação para a Fase 2**: encaixa no dispatcher genérico `ingest-scraping`
— sem Selenium, sem parsing que dependa de JS. Diferente de ANA/ANP (Plone),
precisa de uma `listar_abegas()` própria (paginação por path `/page/N`, não
`?b_start:int=`) e do seletor de conteúdo específico do tema — não dá pra
reaproveitar `listar_ana`/seletores genéricos do dispatcher aqui.